# 02. 自動微分 — jax.grad / value_and_grad / 高階微分

**このノートブックの内容**:
- `jax.grad` で関数を渡すだけで勾配計算
- `value_and_grad` で値と勾配を同時取得
- `jacobian`, `hessian` で多変数微分
- 自動微分で勾配降下法を実装

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_autodiff.md)](../02_autodiff.md)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)

print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
%matplotlib inline

## 1. `jax.grad` — 関数を渡すだけで微分

例: $f(x) = x^3$ の導関数は $f'(x) = 3x^2$。
JAX なら関数 `f` を `jax.grad` に渡すだけ。

In [ ]:
def f(x):
    return x**3

# 手で導関数を書く必要なし
df_dx = jax.grad(f)

print(f'f(2) = {f(2.0)}')                  # 8
print(f"f'(2) = {df_dx(2.0)}  (3·2² = 12)")
print(f"f'(5) = {df_dx(5.0)}  (3·5² = 75)")

## 2. `value_and_grad` — 値と勾配を同時に

勾配降下法では「**現在の損失** と **勾配**」 を両方欲しいので、これが便利。

In [ ]:
def loss(theta):
    return (theta - 3.0) ** 2  # θ=3 で最小値 0

val_and_grad = jax.value_and_grad(loss)

for theta in [0.0, 1.0, 2.5, 3.0, 5.0]:
    val, grad = val_and_grad(theta)
    print(f'θ={theta}: loss={val:.3f}, grad={grad:+.3f}')

## 3. 多変数関数の勾配

$f(\mathbf{x}) = x_0^2 + 2 x_1^2$ の勾配は $\nabla f = (2x_0, 4x_1)$。

In [ ]:
def f(x):
    return x[0]**2 + 2 * x[1]**2

grad_f = jax.grad(f)
x = jnp.array([1.0, 2.0])
print(f'f({x}) = {f(x)}')
print(f'∇f({x}) = {grad_f(x)}  ← (2x₀, 4x₁) = (2, 8)')

## 4. Jacobian と Hessian

- **Jacobian** $J_f$: ベクトル値関数の 1 階微分行列 (`jax.jacrev` / `jax.jacfwd`)
- **Hessian** $H_f$: スカラー関数の 2 階微分行列 (`jax.hessian`)

In [ ]:
def vec_fn(x):
    return jnp.array([x[0]**2 + x[1], x[0] * x[1], x[1]**3])

J = jax.jacrev(vec_fn)
x = jnp.array([1.0, 2.0])
print('Jacobian J_f at x=(1,2):')
print(J(x))
print()

# Hessian (スカラー出力関数のみ)
def scalar_fn(x):
    return x[0]**2 + 2*x[0]*x[1] + 3*x[1]**2

H = jax.hessian(scalar_fn)
print('Hessian H_f:')
print(H(jnp.array([1.0, 2.0])))

## 5. 高階微分 — grad を繰り返し適用

$f(x) = x^4$ なら $f'=4x^3$, $f''=12x^2$, $f'''=24x$, $f''''=24$。

In [ ]:
f = lambda x: x**4

f1 = jax.grad(f)
f2 = jax.grad(f1)
f3 = jax.grad(f2)
f4 = jax.grad(f3)

x0 = 2.0
print(f"f({x0})    = {f(x0)}")
print(f"f'({x0})   = {f1(x0)}  (4·2³  = 32)")
print(f"f''({x0})  = {f2(x0)}  (12·2² = 48)")
print(f"f'''({x0}) = {f3(x0)}  (24·2  = 48)")
print(f"f''''({x0})= {f4(x0)}  (定数 24)")

## 6. 勾配降下法を 5 行で実装

手で微分を書く必要が一切ない。$f(\theta) = (\theta - 3)^2$ の最小化。

In [ ]:
loss = lambda theta: (theta - 3.0) ** 2
grad_loss = jax.grad(loss)

theta = 0.0       # 初期値
lr = 0.1          # 学習率

history = [theta]
for step in range(30):
    theta = theta - lr * grad_loss(theta)
    history.append(float(theta))

print(f'最終 θ = {theta:.4f}  (真の最小点 = 3.0)')
print(f'最終 loss = {loss(theta):.6f}')

plt.figure(figsize=(6, 3))
plt.plot(history, marker='.')
plt.axhline(3.0, color='red', linestyle='--', alpha=0.5, label='真の最適 θ=3')
plt.title('勾配降下法 (jax.grad で自動微分)')
plt.xlabel('ステップ')
plt.ylabel('θ')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [01_basics.ipynb](01_basics.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [03_jit_vmap.ipynb](03_jit_vmap.ipynb) |